# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression. 

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept). 
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already. 
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


In [2]:
NAMEs = "Laine Goldmacher, Max Magrill, Ashley Qin"

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

1. Binary Cross-Entropy (aka log loss), Categorical Cross-Entropy, and Mean Squared Error (MSE) should be used for Binary Classification, Multi-Class Classification, and Regression respectively. 
   1. Binary Cross-Entropy (BCE) measures the difference between predicted probabilities and their actual binary labels (0 or 1).
   2. Categorical Cross-Entropy measures how well the predicted probability distribution matches the true class, penalizing low probabilities assigned to the correct class.
   3. MSE measures the average squared difference between predicted and actual continuous values, penalizing larger errors more heavily.
2. For Binary Classification, there should be one output unit using a sigmoid activation function, producing a probability between 0 and 1. Predictions are classified using a threshold, typically 0.5. For Multi-Class Classification, there should also be one output unit per class using a softmax activation function, producing probabilities that sum to 1. The predicted class is the one with the highest probability. Finally, for Regression, there should still be one output unit with a linear activation, allowing predictions to take any real value.
3. It is important to standardize the data before training each model because standardization ensures that features are on comparable scales, preventing features with larger numerical values from disproportionately influencing gradient updates. This improves the stability and convergence speed of stochastic gradient descent, making training more efficient. It should be fitted on the data before training each model to avoid data leakage; the scaler is fitted on the training data and then applied to both training and holdout sets.
4. Overfitting occurs when a model learns the patterns specific to training data rather than generalizable relationships. We can detect it by comparing training and holdout performance across epochs. If training loss continues decreasing while holdout loss increases, or if training accuracy improves while holdout accuracy stagnates or decreases, this is a sign of overfitting. A large difference in performance between the training and holdout sets is another indicator.

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [25]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

In [19]:
# USE AS MANY CELLS AS NEEDED
#Section 1
def sigmoid(z):
    return 1/(1+torch.exp(-z))

def relu(z):
    return torch.maximum(z, torch.tensor(0.0))

def log_cost(y_pred, y_true):
    # Clamp to avoid log(0)
    eps = torch.finfo(y_pred.dtype).eps
    y_pred = torch.clamp(y_pred, eps, 1 - eps)
    return -torch.mean(y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred))

class BinaryMLP(object):
    
    def __init__(self, 
                 input_dim, 
                 hidden_dim, 
                 learning_rate=0.1):
        
        self.W_1 = torch.randn(input_dim, hidden_dim, requires_grad=True)
        self.b_1 = torch.randn(hidden_dim, requires_grad=True)
        self.W_2 = torch.randn(hidden_dim, 1, requires_grad=True)
        self.b_2 = torch.randn(1, requires_grad=True)
        self.learning_rate = learning_rate
    
    def forward(self, X):
        # 1-input to hidden units 
        h = torch.matmul(X, self.W_1) + self.b_1
        # 2-Find output in each units through their activation function (ReLU in this case)
        h_relu = relu(h)
        # 3- These outputs server as input to the last, output layer consisting of just one unit.
        out = torch.matmul(h_relu, self.W_2) + self.b_2
        return sigmoid(out)
        
    def fit(self, X, y, n_epochs=100):
        
        for epoch in range(n_epochs):

            randperm = torch.randperm(X.shape[0])
            for ii in randperm:
                x_batch, y_batch = X[ii], y[ii]
                
                # Forward pass
                probs = self.forward(x_batch)
                
                # the cost function
                cost = log_cost(probs, y_batch)
            
                # Backward pass 
                cost.backward()
            
                # Gradient descent step (manual)
                with torch.no_grad():
                    self.W_1 -= self.learning_rate * self.W_1.grad
                    self.b_1 -= self.learning_rate * self.b_1.grad
                    self.W_2 -= self.learning_rate * self.W_2.grad
                    self.b_2 -= self.learning_rate * self.b_2.grad
            
                    # Zero gradients
                    self.W_1.grad.zero_()
                    self.b_1.grad.zero_()
                    self.W_2.grad.zero_()
                    self.b_2.grad.zero_()
            
            # Print cost every 10 epochs
            if epoch % 10 == 0:
                print(f"Epoch {epoch}, Cost: {cost.item():.4f}")
        return self    
        
    def predict(self, X):
        with torch.no_grad():

            # Calculate probabilites of input data
            probs = self.forward(X) # COMES HERE (Call foward on X)
            # Assign label  
            predicted = (probs > 0.5).float() # COMES HERE (probs > 0.5 and convert to float)
        return predicted

In [29]:
# Section 2
scalar = StandardScaler()
X_scaled = scalar.fit_transform(X)
X_scaled_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)
X_train, X_report, y_train, y_report = train_test_split(
    X_scaled_tensor, y_tensor, test_size=0.2, random_state=42
)
binaryMLP = BinaryMLP(input_dim=X_scaled_tensor.shape[1], hidden_dim=10, learning_rate=0.1)
binaryMLP.fit(X_train, y_train, n_epochs=50)
y_pred = binaryMLP.predict(X_report)
print(f"Reporting set's accuracy: {accuracy_score(y_report.numpy(), y_pred.numpy()) * 100:.2f}%")

Epoch 0, Cost: 0.7508
Epoch 10, Cost: 0.0004
Epoch 20, Cost: 0.0158
Epoch 30, Cost: 0.0000
Epoch 40, Cost: 0.0021
Reporting set's accuracy: 98.50%


## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3, 
                           n_clusters_per_class=1, 
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [ ]:
# USE AS MANY CELLS AS NEEDED

## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [3]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

YOUR TEXT HERE

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE